# 3.1.4 智能健康监测系统数据分析与优化

## 任务概述

基于 `智能健康监测系统数据集.xlsx`，从以下三方面分析，并生成 **3.1.4-1.docx** 与 **3.1.4-2.docx**。

| 分析维度 | 分析内容 | 目标 |
|----------|----------|------|
| 用户活动周期 | 一天内各时段健康指标变化 | 识别高风险时段与安全时段 |
| 健康指标偏好度 | 血压/血糖/体脂功能使用频率 | 找出受青睐与较少使用的功能 |
| 系统响应与准确性 | 各指标响应时间 | 评估延迟，找误报/延迟关键因素 |

> 运行前请将工作目录切换到本 notebook 所在文件夹。

**配置：** 结果文件保存到**当前目录**（与 xlsx 同级）。

In [ ]:
import os
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt

# plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
# plt.rcParams['axes.unicode_minus'] = False

# OUTPUT_DIR = os.getcwd()
# print('结果保存目录:', OUTPUT_DIR)

**第一步：** 读取数据，预览结构。

In [14]:
FILE = '智能健康监测系统数据集.xlsx'
df = pd.read_excel(FILE)

print('数据规模:', df.shape)
print('列名:', list(df.columns))
print('\n前5行:')
print(df.head())
print('\n各列非空计数:')
print(df.notna().sum())

数据规模: (168, 6)
列名: ['时间戳', '收缩压', '舒张压', '血糖', '体脂分析', '响应时间']

前5行:
                时间戳    收缩压   舒张压    血糖  体脂分析  响应时间
0  2024-09-15 00:00  111.0  80.0   NaN   NaN  0.80
1  2024-09-15 06:00  128.0  79.0   NaN   NaN  0.50
2  2024-09-15 07:00    NaN   NaN  4.37   NaN  0.89
3  2024-09-15 07:00    NaN   NaN   NaN  0.17  0.69
4  2024-09-15 07:30  128.0  74.0   NaN   NaN  0.57

各列非空计数:
时间戳     168
收缩压      80
舒张压      80
血糖       80
体脂分析      8
响应时间    168
dtype: int64


In [16]:
# 时间字段预处理
COL_TIME = '时间戳'
COL_SYS = '收缩压'
COL_DIA = '舒张压'
COL_BS = '血糖'
COL_BF = '体脂分析'
COL_RESP = '响应时间'

df[COL_TIME] = pd.to_datetime(df[COL_TIME])
df['小时'] = df[COL_TIME].dt.hour
df['日期'] = df[COL_TIME].dt.date

print('时间范围:', df[COL_TIME].min(), '~', df[COL_TIME].max())
print('总记录数:', len(df))
print('血压记录:', df[COL_SYS].notna().sum())
print('血糖记录:', df[COL_BS].notna().sum())
print('体脂记录:', df[COL_BF].notna().sum())

时间范围: 2024-09-15 00:00:00 ~ 2024-09-22 22:00:00
总记录数: 168
血压记录: 80
血糖记录: 80
体脂记录: 8


**第二步：** 将宽表转为长表，便于按指标统一分析。

异常判定标准：收缩压≥140 或 舒张压≥90 为血压异常；血糖>7.0 或 <3.9 为血糖异常。

In [17]:
def is_bp_abnormal(row):
    return row[COL_SYS] >= 140 or row[COL_DIA] >= 90

def is_bs_abnormal(val):
    return val > 7.0 or val < 3.9

records = []
for _, r in df.iterrows():
    base = {'时间戳': r[COL_TIME], '小时': r['小时'], COL_RESP: r[COL_RESP]}
    if pd.notna(r[COL_SYS]):
        records.append({**base, '指标': '血压监测', '测量值': r[COL_SYS],
                        '异常': is_bp_abnormal(r)})
    if pd.notna(r[COL_BS]):
        records.append({**base, '指标': '血糖检测', '测量值': r[COL_BS],
                        '异常': is_bs_abnormal(r[COL_BS])})
    if pd.notna(r[COL_BF]):
        records.append({**base, '指标': '体脂分析', '测量值': r[COL_BF], '异常': False})

long_df = pd.DataFrame(records)
print('长表记录数:', len(long_df))
print(long_df.groupby('指标').size())

长表记录数: 168
指标
体脂分析     8
血压监测    80
血糖检测    80
dtype: int64


## 一、用户活动周期（健康指标时段变化）

In [18]:
# 1.1 各小时血压均值
bp_hourly = df.dropna(subset=[COL_SYS]).groupby('小时').agg(
    收缩压均值=(COL_SYS, 'mean'),
    舒张压均值=(COL_DIA, 'mean'),
    测量次数=(COL_SYS, 'count')
).round(2)
print('=== 各小时血压均值 ===')
print(bp_hourly)

bp_peak_hour = bp_hourly['收缩压均值'].idxmax()
bp_peak_val = bp_hourly.loc[bp_peak_hour, '收缩压均值']
print(f'\n收缩压最高时段: {bp_peak_hour} 点，均值 {bp_peak_val} mmHg')

=== 各小时血压均值 ===
     收缩压均值  舒张压均值  测量次数
小时                     
0   110.50  79.88     8
6   113.75  81.38     8
7   131.38  72.38     8
10  115.88  72.50     8
12  118.75  70.25     8
13  115.38  72.50     8
16  114.50  73.00     8
18  106.88  77.00     8
19  110.00  74.38     8
22  111.00  72.00     8

收缩压最高时段: 7 点，均值 131.38 mmHg


In [19]:
# 1.2 各小时血糖均值
bs_hourly = df.dropna(subset=[COL_BS]).groupby('小时').agg(
    血糖均值=(COL_BS, 'mean'),
    测量次数=(COL_BS, 'count')
).round(2)
print('=== 各小时血糖均值 ===')
print(bs_hourly)

bs_peak_hour = bs_hourly['血糖均值'].idxmax()
bs_peak_val = bs_hourly.loc[bs_peak_hour, '血糖均值']
print(f'\n血糖最高时段: {bs_peak_hour} 点，均值 {bs_peak_val} mmol/L')

=== 各小时血糖均值 ===
    血糖均值  测量次数
小时            
7   4.48     8
8   6.90     8
9   5.93     8
12  4.64     8
13  6.81     8
14  6.20     8
18  4.27     8
19  6.81     8
20  5.83     8
22  4.59     8

血糖最高时段: 8 点，均值 6.9 mmol/L


In [20]:
# 1.3 各小时异常率 → 高风险 / 安全时段
risk_hourly = long_df.groupby('小时').agg(
    测量次数=('异常', 'count'),
    异常次数=('异常', 'sum')
)
risk_hourly['异常率(%)'] = (risk_hourly['异常次数'] / risk_hourly['测量次数'] * 100).round(1)
print('=== 各小时异常率 ===')
print(risk_hourly.sort_values('异常率(%)', ascending=False))

high_risk_hours = risk_hourly[risk_hourly['异常率(%)'] >= 10].index.tolist()
safe_hours = risk_hourly[risk_hourly['异常率(%)'] == 0].index.tolist()
print(f'\n高风险时段（异常率≥10%）: {high_risk_hours} 点')
print(f'安全时段（异常率=0%）: {safe_hours} 点')

=== 各小时异常率 ===
    测量次数  异常次数  异常率(%)
小时                    
8      8     3    37.5
13    16     3    18.8
7     24     3    12.5
0      8     1    12.5
19    16     2    12.5
12    16     1     6.2
10     8     0     0.0
9      8     0     0.0
6      8     0     0.0
14     8     0     0.0
16     8     0     0.0
18    16     0     0.0
20     8     0     0.0
22    16     0     0.0

高风险时段（异常率≥10%）: [0, 7, 8, 13, 19] 点
安全时段（异常率=0%）: [6, 9, 10, 14, 16, 18, 20, 22] 点


In [21]:
# 1.4 时段划分汇总
def day_period(h):
    if 6 <= h < 12:
        return '上午(6-12)'
    if 12 <= h < 18:
        return '下午(12-18)'
    if 18 <= h < 22:
        return '傍晚(18-22)'
    return '夜间(0-6/22-24)'

long_df['时段'] = long_df['小时'].apply(day_period)
period_risk = long_df.groupby('时段')['异常'].mean().mul(100).round(1)
period_risk = period_risk.reindex(['上午(6-12)', '下午(12-18)', '傍晚(18-22)', '夜间(0-6/22-24)'])
print('=== 各时段异常率 ===')
print(period_risk)
riskiest_period = period_risk.idxmax()
safest_period = period_risk.idxmin()

=== 各时段异常率 ===
时段
上午(6-12)         10.7
下午(12-18)         8.3
傍晚(18-22)         5.0
夜间(0-6/22-24)     4.2
Name: 异常, dtype: float64


## 二、健康指标偏好度

In [9]:
metric_counts = long_df['指标'].value_counts()
metric_pct = (metric_counts / len(long_df) * 100).round(2)
metric_table = pd.DataFrame({'使用次数': metric_counts, '占比(%)': metric_pct})

print('=== 健康指标偏好度 ===')
print(metric_table)

most_metric = metric_counts.index[0]
least_metric = metric_counts.index[-1]
print(f'\n最受青睐: {most_metric}（{metric_counts.iloc[0]}次, {metric_pct.iloc[0]}%）')
print(f'使用较少: {least_metric}（{metric_counts.iloc[-1]}次, {metric_pct.iloc[-1]}%）')

=== 健康指标偏好度 ===
      使用次数  占比(%)
指标               
血压监测    80  47.62
血糖检测    80  47.62
体脂分析     8   4.76

最受青睐: 血压监测（80次, 47.62%）
使用较少: 体脂分析（8次, 4.76%）


## 三、系统响应与准确性

In [10]:
resp_stats = long_df.groupby('指标')[COL_RESP].agg(
    测量次数='count',
    平均响应='mean',
    中位数='median',
    最大响应='max',
    标准差='std'
).round(3).sort_values('平均响应', ascending=False)

print('=== 各指标响应时间 ===')
print(resp_stats)
print(f'\n全库平均响应时间: {df[COL_RESP].mean():.3f} 秒')

slowest_metric = resp_stats.index[0]
fastest_metric = resp_stats.index[-1]

=== 各指标响应时间 ===
      测量次数   平均响应    中位数  最大响应    标准差
指标                                   
体脂分析     8  0.661  0.660  0.69  0.021
血压监测    80  0.619  0.625  0.99  0.236
血糖检测    80  0.599  0.630  0.98  0.204

全库平均响应时间: 0.612 秒


In [22]:
# 影响指数 = 平均响应 × 使用次数（综合延迟影响）
resp_bottleneck = pd.DataFrame({
    '使用次数': metric_counts,
    '平均响应': resp_stats['平均响应']
})
resp_bottleneck['影响指数'] = (resp_bottleneck['使用次数'] * resp_bottleneck['平均响应']).round(2)
resp_bottleneck = resp_bottleneck.sort_values('影响指数', ascending=False)
print('=== 响应瓶颈分析 ===')
print(resp_bottleneck)
bottleneck_metric = resp_bottleneck.index[0]

=== 响应瓶颈分析 ===
      使用次数   平均响应   影响指数
指标                      
血压监测    80  0.619  49.52
血糖检测    80  0.599  47.92
体脂分析     8  0.661   5.29


In [23]:
# 慢响应记录（>0.85s，可能导致延迟预警）
SLOW_THRESHOLD = 0.85
slow_records = long_df[long_df[COL_RESP] > SLOW_THRESHOLD]
slow_by_metric = slow_records.groupby('指标').size()
slow_by_hour = long_df.groupby('小时')[COL_RESP].mean().sort_values(ascending=False)

print(f'慢响应记录（>{SLOW_THRESHOLD}s）: {len(slow_records)} 条')
print('按指标分布:')
print(slow_by_metric)
print('\n响应较慢的小时 Top5:')
print(slow_by_hour.head(5).round(3))

# 异常测量中的平均响应（误报风险）
abnormal_resp = long_df.groupby('异常')[COL_RESP].mean()
print(f'\n正常测量平均响应: {abnormal_resp.get(False, 0):.3f}s')
print(f'异常测量平均响应: {abnormal_resp.get(True, 0):.3f}s')

慢响应记录（>0.85s）: 26 条
按指标分布:
指标
血压监测    18
血糖检测     8
dtype: int64

响应较慢的小时 Top5:
小时
12    0.668
13    0.665
19    0.637
6     0.630
16    0.622
Name: 响应时间, dtype: float64

正常测量平均响应: 0.611s
异常测量平均响应: 0.617s
